<a href="https://colab.research.google.com/github/samaringle1-arch/Crop-recommendation/blob/main/Crop_Recommendation_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
from google.colab import files

# --- Step 1: Load Your Actual Soil Data ---
print("--- Step 1: Loading Soil Data ---")
print("Please upload your 'soil_dataset.csv' file.")

try:
    uploaded = files.upload()
    file_name = next(iter(uploaded))
    print(f"Reading '{file_name}'...")
    soil_df = pd.read_csv(file_name)

    # --- FIX: Standardize column names ---
    # Rename 'District' to 'district_name' to match the rest of the script.
    soil_df.rename(columns={'District': 'district_name'}, inplace=True)

    # Clean up other column names for easier access (e.g., remove spaces and %)
    soil_df.columns = soil_df.columns.str.strip().str.replace(' %', '_percent').str.replace('%', '_percent')

    print("Soil dataset loaded and columns standardized successfully.")
    print("\n🔹 First 5 rows of your cleaned soil data:")
    print(soil_df.head())

except Exception as e:
    print(f"Error loading soil data: {e}")
    # Create a dummy dataframe to prevent the script from crashing
    soil_df = pd.DataFrame()


# --- Step 2: Fetch Historical Climate Data for Districts in Your Soil File ---
print("\n--- Step 2: Fetching Historical Climate Data ---")
print("Current time: Thursday, September 11, 2025 at 3:44 PM IST")
print("Location: Chandur, Maharashtra, India")


# We need coordinates for the districts. Create a mapping.
# You should expand this list with all districts from your dataset.
DISTRICT_COORDINATES = {
    "Nagpur": {"lat": 21.15, "lon": 79.09}, "Pune": {"lat": 18.52, "lon": 73.86},
    "Nashik": {"lat": 20.00, "lon": 73.79}, "Amravati": {"lat": 20.93, "lon": 77.75},
    "Thane": {"lat": 19.22, "lon": 72.98}, "Mumbai": {"lat": 19.08, "lon": 72.88},
    "Chandur": {"lat": 20.79, "lon": 78.18}, "Anantapur": {"lat": 14.68, "lon": 77.60},
    "Chittoor": {"lat": 13.22, "lon": 79.10}, "East Godavari": {"lat": 16.98, "lon": 82.23},
    "Guntur": {"lat": 16.31, "lon": 80.44}, "Krishna": {"lat": 16.51, "lon": 80.83}
    # Add more districts here...
}

def get_historical_climate_data(districts):
    """
    Fetches average temp and total rainfall for the previous year (2024)
    for a list of districts.
    """
    climate_data_list = []

    for district in districts:
        if district in DISTRICT_COORDINATES:
            coords = DISTRICT_COORDINATES[district]
            lat, lon = coords['lat'], coords['lon']

            print(f"Fetching data for {district}...")

            # API call for daily data for the whole of 2024
            url = (
                f"https://api.open-meteo.com/v1/forecast"
                f"?latitude={lat}&longitude={lon}"
                f"&daily=temperature_2m_mean,precipitation_sum"
                f"&start_date=2024-01-01&end_date=2024-12-31"
            )

            try:
                response = requests.get(url)
                data = response.json()

                if "daily" in data:
                    daily_df = pd.DataFrame(data["daily"])
                    avg_temp = daily_df['temperature_2m_mean'].mean()
                    total_rain = daily_df['precipitation_sum'].sum()

                    climate_data_list.append({
                        'district_name': district,
                        'temperature_celsius': round(avg_temp, 2),
                        'rainfall_mm': round(total_rain, 2)
                    })
                else:
                    print(f"Warning: Could not retrieve daily data for {district}.")

                time.sleep(0.5)

            except Exception as e:
                print(f"Error fetching data for {district}: {e}")
        else:
            print(f"Warning: Coordinates for {district} not found. Skipping.")

    return pd.DataFrame(climate_data_list)

# Check if soil_df is valid before proceeding
if not soil_df.empty and 'district_name' in soil_df.columns:
    districts_to_fetch = soil_df['district_name'].unique()
    climate_df = get_historical_climate_data(districts_to_fetch)
    climate_df['humidity_percent'] = np.random.randint(60, 85, size=len(climate_df))

    print("\n🔹 Fetched and Processed Climate Data:")
    print(climate_df.head())
else:
    print("Cannot fetch climate data because soil DataFrame is empty or missing 'district_name' column.")
    climate_df = pd.DataFrame()


# --- Step 3: Merge Soil and Climate Data ---
print("\n--- Step 3: Merging Datasets ---")
if not soil_df.empty and not climate_df.empty:
    df_merged = pd.merge(soil_df, climate_df, on='district_name', how='left')
    df_merged.dropna(inplace=True)
    print("Merge successful. Merged data sample:")
    print(df_merged.head())
else:
    print("Merge failed. One of the dataframes is empty.")
    df_merged = pd.DataFrame()


# --- Step 4: Create Crop Labels from Rules ---
if not df_merged.empty:
    print("\n--- Step 4: Applying Agricultural Rules to Create Labels ---")

    def assign_crop(row):
        temp = row['temperature_celsius']
        rain = row['rainfall_mm']
        # The soil dataset doesn't have pH, so we remove it from the rules for now
        # or assume a neutral pH. Let's assume neutral.
        ph = 7.0

        if rain > 1000 and 6.0 <= ph <= 7.5 and temp > 25: return 'Rice'
        elif 500 <= rain <= 900 and 15 <= temp <= 22: return 'Wheat'
        elif rain < 500 and temp > 28: return 'Millet'
        elif rain > 1200 and row.get('humidity_percent', 0) > 75 and temp > 26: return 'Sugarcane'
        elif temp > 24 and 600 <= rain <= 1200: return 'Cotton'
        else: return 'Other'

    df_merged['recommended_crop'] = df_merged.apply(assign_crop, axis=1)
    final_df = df_merged[df_merged['recommended_crop'] != 'Other'].copy()

    print("\nFinal dataframe with crop labels (sample):")
    print(final_df.head())
else:
    print("Skipping label creation as merged dataframe is empty.")
    final_df = pd.DataFrame()


# --- Step 5: Preprocess and Train ML Model ---
if not final_df.empty and final_df.shape[0] >= 5: # Lowered threshold slightly for smaller datasets
    print("\n--- Step 5: Preprocessing and Training the Model ---")

    X = final_df.drop('recommended_crop', axis=1)
    y = final_df['recommended_crop']

    categorical_features = ['district_name']
    numerical_features = X.select_dtypes(include=np.number).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ], remainder='passthrough')

    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])

    # Stratify helps ensure train/test sets have similar proportions of each crop
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model_pipeline.fit(X_train, y_train)
    print("\nModel training complete.")

    # --- Step 6: Evaluate, Predict, and Save ---
    print("\n--- Step 6: Evaluating, Predicting, and Saving ---")

    y_pred = model_pipeline.predict(X_test)
    print(f"\nModel Accuracy on Test Set: {accuracy_score(y_test, y_pred):.2f}")

    # Use at most 5 folds, or fewer if there are not enough samples for a class
    cv_folds = min(5, y.value_counts().min())
    if cv_folds > 1:
        print(f"\nPerforming {cv_folds}-fold cross-validation...")
        cv_scores = cross_val_score(model_pipeline, X, y, cv=cv_folds)
        print(f"Average CV Accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std() * 2:.2f})")
    else:
        print("\nSkipping cross-validation because a class has too few samples.")

    print("\nSaving the trained model to 'crop_recommender_model.joblib'...")
    joblib.dump(model_pipeline, 'crop_recommender_model.joblib')
    print("Model saved successfully. You can download it from the Colab file explorer.")

else:
    print("\nSkipping model training: not enough data after applying rules.")



--- Step 1: Loading Soil Data ---
Please upload your 'soil_dataset.csv' file.


Saving soil_dataset.csv to soil_dataset (1).csv
Reading 'soil_dataset (1).csv'...
Soil dataset loaded and columns standardized successfully.

🔹 First 5 rows of your cleaned soil data:
        District  Zn_percent  Fe_percent  Cu_percent  Mn_percent  B_percent  \
0      Anantapur       67.67       65.14       91.88       77.70      73.54   
1       Chittoor       80.51       78.19       99.77       91.82      89.04   
2  East Godavari       79.27       88.14       95.54       97.24      88.05   
3         Guntur       58.30       71.16       98.86       91.40      86.15   
4        Krishna       78.62       82.02       98.05       95.23      65.78   

   S_percent  
0      85.90  
1      88.62  
2      95.67  
3      86.81  
4      98.56  

--- Step 2: Fetching Historical Climate Data ---
Current time: Thursday, September 11, 2025 at 3:44 PM IST
Location: Chandur, Maharashtra, India
Cannot fetch climate data because soil DataFrame is empty or missing 'district_name' column.

--- Step 3: